# Bisexual Lighting — Classifier Validation

Fetches all illustrations tagged `arttag:bisexual-lighting` on Scryfall, saves
the raw API data to `data/existing_tags/`, then runs our HSV classifier against
the subset of those illustrations that exist in the local art folder.

**Metrics reported:**
- **Recall** — what fraction of known bisexual-lighting art our classifier catches
- **False negatives** — known cards the classifier missed
- **False positives** — art in our folder that the classifier flags but Scryfall does not

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

from tools.scryfall_tools import build_session, SF_API_HEADERS
from tools.tagger_tools import create_default_mapping, tag_image_file

DATA_DIR        = Path(os.environ.get("DATA_DIR") or (REPO_ROOT / "data"))
ART_DIR         = DATA_DIR / "art"
TAGS_DIR        = DATA_DIR / "existing_tags"
RAW_OUTPUT_PATH = TAGS_DIR / "bisexual_lighting_scryfall.json"

TAGS_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Data dir: ", DATA_DIR)

## 1 — Fetch all `arttag:bisexual-lighting` illustrations from Scryfall

In [2]:
SEARCH_URL = "https://api.scryfall.com/cards/search"
SEARCH_PARAMS = {"q": "arttag:bisexual-lighting", "unique": "art"}

session = build_session()
all_cards = []
url = SEARCH_URL
params = SEARCH_PARAMS

while url:
    resp = session.get(url, params=params, headers=SF_API_HEADERS)
    resp.raise_for_status()
    page = resp.json()
    all_cards.extend(page["data"])
    print(f"  fetched {len(page['data'])} cards (total so far: {len(all_cards)})")

    if page.get("has_more"):
        url = page["next_page"]
        params = {}          # next_page URL already carries all query params
        time.sleep(0.1)      # Scryfall asks for ~100ms between requests
    else:
        url = None

print(f"\nTotal unique bisexual-lighting illustrations: {len(all_cards)}")

  fetched 175 cards (total so far: 175)
  fetched 145 cards (total so far: 320)

Total unique bisexual-lighting illustrations: 320


## 2 — Save raw API output

In [3]:
raw_output = {
    "query": "arttag:bisexual-lighting",
    "unique": "art",
    "total_cards": len(all_cards),
    "data": all_cards,
}

RAW_OUTPUT_PATH.write_text(json.dumps(raw_output, indent=2), encoding="utf-8")
print(f"Saved {len(all_cards)} records to {RAW_OUTPUT_PATH}")

Saved 320 records to C:\Users\adamc\OneDrive\Documents\GitHub\scryfall-tagger\data\existing_tags\bisexual_lighting_scryfall.json


## 3 — Extract illustration IDs and match against local art

In [4]:
def extract_illustration_ids(cards):
    """Return a dict of illustration_id -> card_name for all known bi-lighting cards."""
    result = {}
    for card in cards:
        # Top-level single-face cards
        if card.get("illustration_id"):
            result[card["illustration_id"]] = card["name"]
        # Double-faced cards — each face has its own illustration
        for face in card.get("card_faces", []):
            if face.get("illustration_id"):
                result[face["illustration_id"]] = f"{card['name']} ({face['name']})"
    return result


known_ids = extract_illustration_ids(all_cards)
print(f"Illustration IDs with bisexual-lighting tag: {len(known_ids)}")

# Build a map of illustration_id -> Path for every file in the art folder
art_by_id = {}
for path in ART_DIR.glob("*.jpg"):
    # Filename format: CardName_<uuid>.jpg  — UUID is the last _-separated segment
    stem = path.stem
    parts = stem.rsplit("_", 1)
    if len(parts) == 2:
        art_by_id[parts[1]] = path

print(f"Art files in local folder:               {len(art_by_id)}")

# Intersection
matched_ids = set(known_ids.keys()) & set(art_by_id.keys())
print(f"Known bisexual-lighting art found locally: {len(matched_ids)} / {len(known_ids)}")

Illustration IDs with bisexual-lighting tag: 328
Art files in local folder:               52891
Known bisexual-lighting art found locally: 328 / 328


## 4 — Run classifier on matched art

In [5]:
from tqdm.notebook import tqdm

mapping = create_default_mapping()

rows = []
for ill_id in tqdm(sorted(matched_ids), desc="Classifying"):
    path = art_by_id[ill_id]
    try:
        selected, top_score, scores = tag_image_file(path, mapping)
    except ValueError as exc:
        print(f"  SKIP {path.name}: {exc}")
        continue

    tag_ids = [t["id"] for t in selected]
    bi_score = scores.get("bisexual-lighting", 0.0)
    detected = "bisexual-lighting" in tag_ids

    rows.append({
        "illustration_id": ill_id,
        "card_name": known_ids[ill_id],
        "file": path.name,
        "bi_score": round(bi_score, 4),
        "detected": detected,
        "top_colour": tag_ids[0] if tag_ids else None,
        "all_tags": ", ".join(tag_ids),
    })

df = pd.DataFrame(rows)
print(f"Classified {len(df)} images")

Classifying:   0%|          | 0/328 [00:00<?, ?it/s]

Classified 328 images


## 5 — Recall and false negatives

In [6]:
n_total    = len(df)
n_detected = df["detected"].sum()
recall     = n_detected / n_total if n_total else 0

print(f"Recall:  {n_detected} / {n_total}  ({recall:.1%})")
print()

false_negatives = df[~df["detected"]].sort_values("bi_score", ascending=False)
print(f"False negatives ({len(false_negatives)}):")
false_negatives[["card_name", "bi_score", "top_colour", "all_tags"]].head(20)

Recall:  209 / 328  (63.7%)

False negatives (119):


,card_name,bi_score,top_colour,all_tags
199,Crosis's Charm,0.0299,black,"black, purple, dark-purple, blue, navy"
94,Scalding Tarn,0.0295,red,"red, dark-pink, pink, dark-red, purple"
269,Scrabbling Claws,0.0290,navy,"navy, blue, black, dark-purple, purple"
263,Facet Reader,0.0290,navy,"navy, blue, black, light-blue, purple"
25,"Shantotto, Tactician Magician",0.0288,blue,"blue, light-blue, navy, purple, black"
323,Mountain,0.0285,red,"red, dark-pink, purple, dark-red, pink"
116,Ashnod's Altar,0.0276,navy,"navy, blue, light-blue, purple, black"
292,Bösium Strip,0.0275,red,"red, pink, light-red, purple, grey"
76,Blade-Blizzard Kitsune,0.0271,blue,"blue, navy, purple, lavender, light-blue"
119,Magus of the Moon,0.0269,red,"red, dark-red, black, light-red, peach"


## 6 — Score distribution

In [7]:
print("bi_score distribution for known bisexual-lighting art:")
print(df["bi_score"].describe().round(4))
print()

buckets = pd.cut(df["bi_score"], bins=[0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20, 1.0])
print(df.groupby(buckets, observed=True)["detected"].agg(["count", "sum"]).rename(columns={"sum": "detected"}))

bi_score distribution for known bisexual-lighting art:
count    328.0000
mean       0.0758
std        0.0747
min        0.0000
25%        0.0131
50%        0.0541
75%        0.1101
max        0.3574
Name: bi_score, dtype: float64

              count  detected
bi_score                     
(0.0, 0.02]      79         0
(0.02, 0.04]     42        18
(0.04, 0.06]     35        35
(0.06, 0.08]     37        37
(0.08, 0.1]      26        26
(0.1, 0.15]      32        32
(0.15, 0.2]      37        37
(0.2, 1.0]       24        24


## 7 — False positives (art in local folder the classifier flags that Scryfall does not)

In [ ]:
# Sample up to 200 non-bisexual-lighting files for a spot-check
import random

non_bi_ids = sorted(set(art_by_id.keys()) - set(known_ids.keys()))
sample_ids = random.sample(non_bi_ids, min(200, len(non_bi_ids)))

fp_rows = []
for ill_id in tqdm(sample_ids, desc="False-positive scan"):
    path = art_by_id[ill_id]
    try:
        selected, _, scores = tag_image_file(path, mapping)
    except ValueError:
        continue
    tag_ids = [t["id"] for t in selected]
    if "bisexual-lighting" in tag_ids:
        fp_rows.append({
            "illustration_id": ill_id,
            "file": path.name,
            "bi_score": round(scores.get("bisexual-lighting", 0), 4),
            "all_tags": ", ".join(tag_ids),
        })

fp_df = pd.DataFrame(fp_rows)
fp_rate = len(fp_df) / len(sample_ids) if sample_ids else 0
print(f"False positive rate in sample: {len(fp_df)} / {len(sample_ids)}  ({fp_rate:.1%})")
if not fp_df.empty:
    fp_df.sort_values("bi_score", ascending=False).head(20)

False-positive scan:   0%|          | 0/200 [00:00<?, ?it/s]

False positive rate in sample: 10 / 200  (5.0%)


: 